In [2]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("dataturks/resume-entities-for-ner")

print("Path to dataset files:", path)

c:\Users\ASUS\anaconda3\envs\ml_env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


100%|██████████| 323k/323k [00:01<00:00, 270kB/s]

Extracting files...
Path to dataset files: C:\Users\ASUS\.cache\kagglehub\datasets\dataturks\resume-entities-for-ner\versions\1


In [3]:
# ============================================================
# RESUME NER — Google Colab Training Notebook
# Dataset: dataturks/resume-entities-for-ner (via kagglehub)
# Model:   bert-base-uncased → fine-tuned for token classification
# Entities: Name, Email Address, Skills, Designation, Degree,
#            College Name, Companies worked at, Location,
#            Graduation Year, Years of Experience
# ============================================================


# ─── CELL 1: Install All Dependencies ───────────────────────
# ⚠️ Runtime may restart after this — re-run from Cell 2 if it does
!pip install kagglehub transformers datasets seqeval evaluate torch --quiet
print("✅ All packages installed")


# ─── CELL 2: Imports ────────────────────────────────────────
import os
import json
import random
import numpy as np
import torch
from collections import defaultdict, Counter

from transformers import (
    AutoTokenizer,
    AutoModelForTokenClassification,
    TrainingArguments,
    Trainer,
    DataCollatorForTokenClassification,
    pipeline,
)
from datasets import Dataset
import evaluate

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

print("✅ All imports successful")
print(f"🔧 Device: {'GPU ✅' if torch.cuda.is_available() else 'CPU ⚠️  (GPU strongly recommended — Runtime > Change runtime type > T4 GPU)'}")


# ─── CELL 3: Kaggle Authentication ──────────────────────────
# Get your API key from: kaggle.com → Account → Create New API Token
# It downloads a kaggle.json — copy the username and key from it

os.environ["KAGGLE_USERNAME"] = "your_kaggle_username"   # ← replace
os.environ["KAGGLE_KEY"]      = "your_kaggle_api_key"    # ← replace

# Alternative: upload kaggle.json directly to Colab and run:
# import json
# with open("/root/.kaggle/kaggle.json") as f:
#     creds = json.load(f)
# os.environ["KAGGLE_USERNAME"] = creds["username"]
# os.environ["KAGGLE_KEY"]      = creds["key"]

print("✅ Kaggle credentials set")


# ─── CELL 4: Download Dataset via kagglehub ─────────────────
import kagglehub

path = kagglehub.dataset_download("dataturks/resume-entities-for-ner")
print("📁 Path to dataset files:", path)

# List files in downloaded folder
files = os.listdir(path)
print("📄 Files found:", files)

# Auto-pick the JSON file
JSON_PATH = os.path.join(path, [f for f in files if f.endswith(".json")][0])
print("✅ Using file:", JSON_PATH)


# ─── CELL 5: Load & Inspect Dataset ─────────────────────────
def load_jsonl(path):
    """Load JSONL — one JSON object per line."""
    records = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                records.append(json.loads(line))
    return records

raw_data = load_jsonl(JSON_PATH)
print(f"📊 Total records loaded: {len(raw_data)}")

# Show all unique entity labels present in dataset
all_labels = set()
for r in raw_data:
    for ann in r.get("annotation", []):
        for lbl in ann.get("label", []):
            all_labels.add(lbl)

print(f"🏷️  Entity types found: {sorted(all_labels)}")

# Show first record as sanity check
print("\n--- Sample Record (first resume, first 500 chars) ---")
print(raw_data[0]["content"][:500])
print("\n--- Annotations ---")
print(json.dumps(raw_data[0]["annotation"][:3], indent=2))


# ─── CELL 6: Define Label Schema (BIO Tagging) ──────────────
# Drop 'UNKNOWN' — it is noise, not a useful entity
ENTITY_TYPES = [
    "Name",
    "Email Address",
    "Skills",
    "Designation",
    "Degree",
    "College Name",
    "Companies worked at",
    "Location",
    "Graduation Year",
    "Years of Experience",
]

# BIO scheme: O, B-EntityType, I-EntityType
label_list = ["O"]
for ent in ENTITY_TYPES:
    label_list.append(f"B-{ent}")
    label_list.append(f"I-{ent}")

label2id = {l: i for i, l in enumerate(label_list)}
id2label = {i: l for l, i in label2id.items()}

print(f"📊 Total label classes: {len(label_list)}")
for lbl in label_list:
    print(f"   {label2id[lbl]:>2}  {lbl}")


# ─── CELL 7: Load Tokenizer ─────────────────────────────────
MODEL_NAME = "bert-base-uncased"
tokenizer  = AutoTokenizer.from_pretrained(MODEL_NAME)
print(f"✅ Tokenizer loaded: {MODEL_NAME}")


# ─── CELL 8: Convert Span Annotations → BIO Token Labels ────
def span_to_bio(text, annotations, tokenizer, label2id, max_length=512):
    """
    Convert character-level span annotations to BIO token labels.

    Pipeline:
      1. Build a character-level label array from annotation spans
      2. Tokenize with offset_mapping to map chars → subword tokens
      3. Assign BIO label to each token using its char start position
      4. Mark special tokens ([CLS], [SEP]) with -100 (ignored in loss)
    """
    # Step 1: char-level label array (default = "O")
    char_labels = ["O"] * len(text)

    for ann in annotations:
        entity_type = ann["label"][0]
        if entity_type == "UNKNOWN" or entity_type not in ENTITY_TYPES:
            continue
        for point in ann["points"]:
            start = point["start"]
            end   = min(point["end"] + 1, len(text))  # end is inclusive in this dataset
            for i in range(start, end):
                if i < len(char_labels) and char_labels[i] == "O":
                    char_labels[i] = ("B-" if i == start else "I-") + entity_type

    # Step 2: tokenize with offset mapping
    encoding = tokenizer(
        text,
        max_length=max_length,
        truncation=True,
        return_offsets_mapping=True,
        padding=False,
    )
    offsets = encoding["offset_mapping"]

    # Step 3: align char labels → token labels
    token_labels = []
    for (tok_start, tok_end) in offsets:
        if tok_start == tok_end:       # special token [CLS] / [SEP]
            token_labels.append(-100)
        else:
            char_label = char_labels[tok_start]
            token_labels.append(label2id.get(char_label, label2id["O"]))

    encoding["labels"] = token_labels
    encoding.pop("offset_mapping")
    return encoding


# Process all records
processed = []
skipped   = 0
for r in raw_data:
    try:
        enc = span_to_bio(
            r["content"],
            r.get("annotation", []),
            tokenizer,
            label2id,
        )
        processed.append(enc)
    except Exception as e:
        skipped += 1

print(f"✅ Processed: {len(processed)} records")
print(f"⚠️  Skipped:   {skipped} records (malformed)")

# Entity distribution in processed data
entity_counts = Counter()
for rec in processed:
    for lbl_id in rec["labels"]:
        if lbl_id not in (-100, label2id["O"]):
            entity_counts[id2label[lbl_id].replace("B-", "").replace("I-", "")] += 1

print("\n📊 Entity token counts in dataset:")
for ent, cnt in entity_counts.most_common():
    print(f"   {ent:<28} {cnt:>6} tokens")


# ─── CELL 9: Train / Val / Test Split ───────────────────────
random.shuffle(processed)

n       = len(processed)
n_train = int(0.70 * n)
n_val   = int(0.15 * n)
n_test  = n - n_train - n_val

train_data = processed[:n_train]
val_data   = processed[n_train: n_train + n_val]
test_data  = processed[n_train + n_val:]

print(f"📊 Train: {len(train_data)} | Val: {len(val_data)} | Test: {len(test_data)}")


def to_hf_dataset(records):
    """Convert list of tokenizer output dicts → HuggingFace Dataset."""
    batch = defaultdict(list)
    for rec in records:
        for k, v in rec.items():
            batch[k].append(v)
    return Dataset.from_dict(dict(batch))

train_ds = to_hf_dataset(train_data)
val_ds   = to_hf_dataset(val_data)
test_ds  = to_hf_dataset(test_data)

print("✅ HuggingFace Datasets created")
print(f"   Train columns: {train_ds.column_names}")


# ─── CELL 10: Load Model ────────────────────────────────────
model = AutoModelForTokenClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(label_list),
    id2label=id2label,
    label2id=label2id,
    ignore_mismatched_sizes=True,
)

total_params     = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"✅ Model loaded: {MODEL_NAME}")
print(f"   Total params    : {total_params:,}")
print(f"   Trainable params: {trainable_params:,}")


# ─── CELL 11: Metrics Setup ─────────────────────────────────
seqeval = evaluate.load("seqeval")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)

    true_labels_batch, true_preds_batch = [], []
    for pred_seq, label_seq in zip(predictions, labels):
        tl, tp = [], []
        for p, l in zip(pred_seq, label_seq):
            if l == -100:
                continue
            tl.append(id2label[l])
            tp.append(id2label[p])
        true_labels_batch.append(tl)
        true_preds_batch.append(tp)

    results = seqeval.compute(
        predictions=true_preds_batch,
        references=true_labels_batch,
        zero_division=0,
    )

    flat = {
        "precision": results["overall_precision"],
        "recall":    results["overall_recall"],
        "f1":        results["overall_f1"],
        "accuracy":  results["overall_accuracy"],
    }
    # Per-entity F1 logged separately
    for ent in ENTITY_TYPES:
        if ent in results:
            flat[f"f1_{ent.replace(' ', '_')}"] = results[ent]["f1"]

    return flat

print("✅ Metrics function ready (seqeval — entity-level Precision / Recall / F1)")


# ─── CELL 12: Training Arguments ────────────────────────────
OUTPUT_DIR = "./resume-ner-bert"

training_kwargs = dict(
    output_dir                  = OUTPUT_DIR,
    num_train_epochs            = 8,
    per_device_train_batch_size = 8,
    per_device_eval_batch_size  = 8,
    learning_rate               = 2e-5,
    weight_decay                = 0.01,
    warmup_ratio                = 0.1,
    lr_scheduler_type           = "linear",
    save_strategy               = "epoch",
    load_best_model_at_end      = True,
    metric_for_best_model       = "f1",
    greater_is_better           = True,
    logging_dir                 = "./logs",
    logging_steps               = 10,
    report_to                   = "none",
    fp16                        = torch.cuda.is_available(),
    seed                        = SEED,
)

# transformers>=4.46 renamed evaluation_strategy -> eval_strategy
try:
    training_args = TrainingArguments(evaluation_strategy="epoch", **training_kwargs)
except TypeError:
    training_args = TrainingArguments(eval_strategy="epoch", **training_kwargs)

data_collator = DataCollatorForTokenClassification(
    tokenizer,
    padding=True,
    pad_to_multiple_of=8 if torch.cuda.is_available() else None,
)

trainer_kwargs = dict(
    model           = model,
    args            = training_args,
    train_dataset   = train_ds,
    eval_dataset    = val_ds,
    data_collator   = data_collator,
    compute_metrics = compute_metrics,
)

# Trainer API changed across versions: tokenizer -> processing_class
try:
    trainer = Trainer(tokenizer=tokenizer, **trainer_kwargs)
except TypeError:
    try:
        trainer = Trainer(processing_class=tokenizer, **trainer_kwargs)
    except TypeError:
        trainer = Trainer(**trainer_kwargs)

print("✅ Trainer configured")
print(f"   Epochs        : {training_args.num_train_epochs}")
print(f"   Batch size    : {training_args.per_device_train_batch_size}")
print(f"   Learning rate : {training_args.learning_rate}")
print(f"   FP16 (mixed)  : {training_args.fp16}")


# ─── CELL 13: Train ─────────────────────────────────────────
print("\n🚀 Starting training...\n")
train_result = trainer.train()

print("\n" + "="*60)
print("🏁 TRAINING COMPLETE")
print(f"   Total steps   : {train_result.global_step}")
print(f"   Training loss : {train_result.training_loss:.4f}")
print(f"   Time (s)      : {train_result.metrics['train_runtime']:.1f}")
print("="*60)


# ─── CELL 14: Evaluate on Validation Set ────────────────────
print("\n📊 VALIDATION SET METRICS")
print("-"*55)
val_metrics = trainer.evaluate(val_ds)
for k, v in sorted(val_metrics.items()):
    if not k.startswith("eval_runtime") and not k.startswith("eval_samples"):
        label = k.replace("eval_", "").replace("_", " ").capitalize()
        print(f"  {label:<40} {v:.4f}")


# ─── CELL 15: Evaluate on Test Set ──────────────────────────
print("\n📊 TEST SET METRICS")
print("-"*55)
test_metrics = trainer.evaluate(test_ds)
for k, v in sorted(test_metrics.items()):
    if not k.startswith("eval_runtime") and not k.startswith("eval_samples"):
        label = k.replace("eval_", "").replace("_", " ").capitalize()
        print(f"  {label:<40} {v:.4f}")


# ─── CELL 16: Full Per-Entity Classification Report ─────────
def detailed_entity_report(trainer, dataset, split_name="Test"):
    """Print a full entity-level Precision / Recall / F1 table."""
    out    = trainer.predict(dataset)
    preds  = np.argmax(out.predictions, axis=-1)
    labels = out.label_ids

    true_labels_batch, true_preds_batch = [], []
    for pred_seq, label_seq in zip(preds, labels):
        tl, tp = [], []
        for p, l in zip(pred_seq, label_seq):
            if l == -100:
                continue
            tl.append(id2label[l])
            tp.append(id2label[p])
        true_labels_batch.append(tl)
        true_preds_batch.append(tp)

    results = seqeval.compute(
        predictions=true_preds_batch,
        references=true_labels_batch,
        zero_division=0,
    )

    print(f"\n{'='*70}")
    print(f"  DETAILED NER REPORT — {split_name} Set")
    print(f"{'='*70}")
    print(f"  {'Entity':<28} {'Precision':>10} {'Recall':>10} {'F1':>10} {'Support':>10}")
    print(f"  {'-'*66}")

    for ent in ENTITY_TYPES:
        if ent in results:
            r = results[ent]
            print(f"  {ent:<28} {r['precision']:>10.4f} {r['recall']:>10.4f} "
                  f"{r['f1']:>10.4f} {r['number']:>10}")
        else:
            print(f"  {ent:<28} {'N/A':>10} {'N/A':>10} {'N/A':>10} {'0':>10}")

    print(f"  {'-'*66}")
    print(f"  {'OVERALL (entity-level)':<28} "
          f"{results['overall_precision']:>10.4f} "
          f"{results['overall_recall']:>10.4f} "
          f"{results['overall_f1']:>10.4f}")
    print(f"  {'Token Accuracy':<28} {results['overall_accuracy']:>10.4f}")
    print(f"{'='*70}")

    return results

test_results = detailed_entity_report(trainer, test_ds, "Test")
_            = detailed_entity_report(trainer, val_ds,  "Validation")


# ─── CELL 17: Confusion Analysis ────────────────────────────
def entity_confusion(trainer, dataset, top_n=15):
    """Show the most common entity mis-classifications."""
    out    = trainer.predict(dataset)
    preds  = np.argmax(out.predictions, axis=-1)
    labels = out.label_ids

    confusion = Counter()
    for pred_seq, label_seq in zip(preds, labels):
        for p, l in zip(pred_seq, label_seq):
            if l == -100:
                continue
            true_tag = id2label[l]
            pred_tag = id2label[p]
            if true_tag != pred_tag:
                confusion[(true_tag, pred_tag)] += 1

    print(f"\n🔀 TOP {top_n} MOST CONFUSED ENTITY PAIRS (Test Set)")
    print(f"  {'True Label':<32} {'Predicted As':<32} {'Count':>8}")
    print(f"  {'-'*74}")
    for (true_l, pred_l), count in confusion.most_common(top_n):
        print(f"  {true_l:<32} {pred_l:<32} {count:>8}")

entity_confusion(trainer, test_ds)


# ─── CELL 18: Inference on a New Resume ─────────────────────
ner_pipeline = pipeline(
    "ner",
    model=model,
    tokenizer=tokenizer,
    aggregation_strategy="simple",   # merges B- / I- tokens automatically
    device=0 if torch.cuda.is_available() else -1,
)

SAMPLE_RESUME = """
Heet Shah
heet.shah@gmail.com
Mumbai, Maharashtra, India

Software Engineer at TCS | June 2022 – Present
- Built REST APIs using Python and Django
- Deployed microservices with Docker and Kubernetes
- Managed PostgreSQL and MySQL databases

Education:
B.Tech in Computer Science Engineering
VJTI Mumbai — Graduated 2022

Skills: Python, Django, FastAPI, Docker, PostgreSQL,
        Git, Machine Learning, NLP, REST APIs

Years of Experience: 2 years
"""

print("\n🔍 INFERENCE ON SAMPLE RESUME")
print("-"*60)
ner_results = ner_pipeline(SAMPLE_RESUME)
for r in ner_results:
    print(f"  [{r['entity_group']:<28}]  \"{r['word']}\"  (conf: {r['score']:.3f})")


# ─── CELL 19: Save Model ────────────────────────────────────
SAVE_PATH = "./resume-ner-final"
trainer.save_model(SAVE_PATH)
tokenizer.save_pretrained(SAVE_PATH)
print(f"\n💾 Model saved to '{SAVE_PATH}'")
print("   To reload in a new session:")
print("   from transformers import pipeline")
print(f"   ner = pipeline('ner', model='{SAVE_PATH}', aggregation_strategy='simple')")

# Optional: save to Google Drive so it persists after Colab session ends
# from google.colab import drive
# drive.mount('/content/drive')
# import shutil
# shutil.copytree(SAVE_PATH, '/content/drive/MyDrive/resume-ner-final')
# print("✅ Model also saved to Google Drive")


# ─── CELL 20: Final Summary Dashboard ───────────────────────
print("\n" + "="*65)
print("  📈 FINAL PERFORMANCE SUMMARY")
print("="*65)
print(f"  Base model     : {MODEL_NAME}")
print(f"  Train samples  : {len(train_data)}")
print(f"  Val samples    : {len(val_data)}")
print(f"  Test samples   : {len(test_data)}")
print(f"  Epochs         : {training_args.num_train_epochs}")
print(f"  Learning rate  : {training_args.learning_rate}")
print(f"  Batch size     : {training_args.per_device_train_batch_size}")
print(f"  Mixed FP16     : {training_args.fp16}")
print()
print(f"  {'Metric':<22} {'Validation':>12} {'Test':>12}")
print(f"  {'-'*48}")
for metric in ["eval_f1", "eval_precision", "eval_recall", "eval_accuracy"]:
    v = val_metrics.get(metric, 0)
    t = test_metrics.get(metric, 0)
    label = metric.replace("eval_", "").capitalize()
    print(f"  {label:<22} {v:>12.4f} {t:>12.4f}")
print("="*65)

print("""
💡 TIPS TO IMPROVE ACCURACY:
─────────────────────────────────────────────────────────────
 1. Use 'dslim/bert-base-NER' as base model instead of
    bert-base-uncased — it already knows NER, giving a
    head start and typically +3-5% F1

 2. Increase epochs to 10-15 if val_f1 is still climbing
    at epoch 8 (watch for overfitting via val loss rising)

 3. Try learning_rate = 3e-5 if the model underfits,
    or 1e-5 if it overfits

 4. Annotate more resumes — 500+ is ideal for production.
    Use Label Studio (free) to annotate quickly.

 5. Try roberta-base instead of bert-base for ~2% more F1
    (slower to train but more accurate)

 6. Add a CRF layer on top of BERT for better sequence
    consistency — especially helps with B/I tag ordering
─────────────────────────────────────────────────────────────
""")

✅ All packages installed
✅ All imports successful
🔧 Device: CPU ⚠️  (GPU strongly recommended — Runtime > Change runtime type > T4 GPU)
✅ Kaggle credentials set
📁 Path to dataset files: C:\Users\ASUS\.cache\kagglehub\datasets\dataturks\resume-entities-for-ner\versions\1
📄 Files found: ['Entity Recognition in Resumes.json']
✅ Using file: C:\Users\ASUS\.cache\kagglehub\datasets\dataturks\resume-entities-for-ner\versions\1\Entity Recognition in Resumes.json
📊 Total records loaded: 220
🏷️  Entity types found: ['College Name', 'Companies worked at', 'Degree', 'Designation', 'Email Address', 'Graduation Year', 'Location', 'Name', 'Skills', 'UNKNOWN', 'Years of Experience']

--- Sample Record (first resume, first 500 chars) ---
Abhishek Jha
Application Development Associate - Accenture

Bengaluru, Karnataka - Email me on Indeed: indeed.com/r/Abhishek-Jha/10e7a8cb732bc43a

• To work for an organization which provides me the opportunity to improve my skills
and knowledge for my individual and c

Loading weights: 100%|██████████| 197/197 [00:00<00:00, 7709.11it/s]
BertForTokenClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
bert.pooler.dense.bias                     | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
bert.pooler.dense.weight                   | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you exp

✅ Model loaded: bert-base-uncased
   Total params    : 108,907,797
   Trainable params: 108,907,797


warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.
c:\Users\ASUS\anaconda3\envs\ml_env\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


✅ Metrics function ready (seqeval — entity-level Precision / Recall / F1)
✅ Trainer configured
   Epochs        : 8
   Batch size    : 8
   Learning rate : 2e-05
   FP16 (mixed)  : False

🚀 Starting training...



Epoch,Training Loss,Validation Loss,Precision,Recall,F1,Accuracy,F1 Name,F1 Email Address,F1 Skills,F1 Designation,F1 Degree,F1 College Name,F1 Companies Worked At,F1 Location,F1 Graduation Year,F1 Years Of Experience
1,2.941077,1.004402,0.000000,0.000000,0.000000,0.797632,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
2,0.858573,0.692282,0.013793,0.005405,0.007767,0.831942,0.000000,0.026144,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
3,0.608912,0.548410,0.036765,0.013514,0.019763,0.838790,0.000000,0.097087,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
4,0.515076,0.475624,0.143969,0.100000,0.118022,0.856409,0.344828,0.365385,0.000000,0.076923,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
5,0.456571,0.418554,0.135897,0.143243,0.139474,0.874670,0.523810,0.413043,0.000000,0.240000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
6,0.375820,0.421434,0.162471,0.191892,0.175960,0.876168,0.471910,0.420000,0.000000,0.238806,0.000000,0.086022,0.131868,0.120000,0.000000,0.000000
7,0.381207,0.400801,0.184652,0.208108,0.195680,0.879307,0.575000,0.395833,0.000000,0.296296,0.052632,0.050000,0.136364,0.218182,0.000000,0.000000
8,0.376375,0.397770,0.188862,0.210811,0.199234,0.878736,0.582278,0.426966,0.000000,0.285714,0.051282,0.048193,0.173913,0.214286,0.000000,0.000000


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.69it/s]
c:\Users\ASUS\anaconda3\envs\ml_env\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  3.76it/s]
c:\Users\ASUS\anaconda3\envs\ml_env\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  2.95it/s]
c:\Users\ASUS\anaconda3\envs\ml_env\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  3.22it/s]
c:\Users\ASUS\ana


🏁 TRAINING COMPLETE
   Total steps   : 152
   Training loss : 0.7071
   Time (s)      : 2170.7

📊 VALIDATION SET METRICS
-------------------------------------------------------


RuntimeError: on_train_begin must be called before on_evaluate

In [4]:
trainer.predict(val_ds)

PredictionOutput(predictions=array([[[ 2.1017959e+00, -3.2308078e-01, -4.3532169e-01, ...,
         -8.7683457e-01, -9.7715074e-01, -6.5461755e-01],
        [-3.1744313e-01,  7.5521874e-01,  8.9069188e-01, ...,
         -5.7880461e-01, -1.1023508e-01, -1.3102263e-01],
        [-1.3638034e-01,  7.0340490e-01,  4.1707745e+00, ...,
         -9.5876670e-01, -1.0919907e+00, -6.5591246e-01],
        ...,
        [ 2.4647110e+00, -9.7212559e-01, -4.2231435e-01, ...,
         -1.1276383e+00, -8.2508343e-01, -2.5565606e-01],
        [ 2.4150786e+00, -9.8541123e-01, -5.2613753e-01, ...,
         -1.0645528e+00, -8.2548487e-01, -2.9186961e-01],
        [ 2.7764537e+00, -1.1829979e+00, -7.7740431e-01, ...,
         -8.9853895e-01, -8.3130252e-01, -4.0463042e-01]],

       [[ 2.7583780e+00, -2.0989908e-01, -5.3033525e-01, ...,
         -8.0502129e-01, -8.9977133e-01, -7.5898147e-01],
        [-3.3160818e-01,  8.9966804e-01,  1.3391941e+00, ...,
         -6.4377475e-01, -1.9048719e-01, -1.3628137e-0

In [5]:
trainer.evaluate()

RuntimeError: on_train_begin must be called before on_evaluate

In [12]:
from transformers import AutoModelForTokenClassification

model = AutoModelForTokenClassification.from_pretrained(
    r"C:\Users\ASUS\Documents\Study\NLP\Project\resume-ner-bert\resume-ner-final"
)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 4535.89it/s]


In [25]:
from transformers import AutoTokenizer, pipeline

tokenizer = AutoTokenizer.from_pretrained(
    r"C:\Users\ASUS\Documents\Study\NLP\Project\resume-ner-bert\resume-ner-final"
)

ner = pipeline(
    "ner",
    model=model,
    tokenizer=tokenizer,
    aggregation_strategy="simple"
)

In [23]:
resume  = """
Krish Sodhi
+91 74991 97208 | sodhi.krish05@gmail.com | github/k-r-i-s-h-7
EDUCATION
B.Tech in Computer Engineering Aug 2023 – June 2027
Sardar Patel Institute of Technology, Mumbai, India
CGPA: 8.16
Minors: UI/UX Designing
PROJECTS
1. Mock Interviewer Platform | Node.js, Redis, LLM APIs, WebSockets, Speech-to-Text, Auth,
Docker, Cloud
⋄ Built an end-to-end AI-driven mock interview platform with real-time voice-based technical and be-
havioral interviews.
⋄ Implemented secure authentication and user session management for personalized interview tracking.
⋄ Integrated Redis caching to manage interview state and reduce repeated LLM computation.
⋄ Developed a real-time interview flow using speech-to-text and WebSockets for dynamic questioning.
⋄ Automated JD- and resume-aware evaluation to generate structured feedback and selection verdicts.
⋄ Deployed the system using Docker and cloud infrastructure for production readiness.
2. Wave: AI Website & App Generator | AI/LLMs, Gluestack, React Native, Expo, Shadcn,
Tailwind CSS, Vite, Firebase, Git
⋄ DevelopedanAI-poweredplatformgeneratingfullyfunctionalfrontendcodeforbothmobileappsand
websites.
⋄ Utilized Gluestack, React Native, and Expo for robust mobile app frontend generation.
⋄ Leveraged Shadcn, Tailwind CSS, and Vite for efficient and modern web frontend development.
⋄ IntegratedautomatedGitversioningforallgeneratedprojects,enablingeasycollaborationandchange
tracking.
...
Generative AI Langchain, LangGraph, CrewAI, Langflow, RAG, Agentic AI
ACHIEVEMENTS
Hackathon Participations: SE Hack SPIT, Cyber Cypher NMIMS Hackathon
1"""

In [21]:
import pdfplumber

with pdfplumber.open(r"Parsing\Resume\Resume (2).pdf") as pdf:
    text = "\n".join(page.extract_text() for page in pdf.pages)

print(text)

Krish Sodhi
+91 74991 97208 | sodhi.krish05@gmail.com | github/k-r-i-s-h-7
EDUCATION
B.Tech in Computer Engineering Aug 2023 – June 2027
Sardar Patel Institute of Technology, Mumbai, India
CGPA: 8.16
Minors: UI/UX Designing
PROJECTS
1. Mock Interviewer Platform | Node.js, Redis, LLM APIs, WebSockets, Speech-to-Text, Auth,
Docker, Cloud
⋄ Built an end-to-end AI-driven mock interview platform with real-time voice-based technical and be-
havioral interviews.
⋄ Implemented secure authentication and user session management for personalized interview tracking.
⋄ Integrated Redis caching to manage interview state and reduce repeated LLM computation.
⋄ Developed a real-time interview flow using speech-to-text and WebSockets for dynamic questioning.
⋄ Automated JD- and resume-aware evaluation to generate structured feedback and selection verdicts.
⋄ Deployed the system using Docker and cloud infrastructure for production readiness.
2. Wave: AI Website & App Generator | AI/LLMs, Gluestack, React

In [26]:


results = ner(resume)

for r in results:
    print(r)

{'entity_group': 'Name', 'score': np.float32(0.36966988), 'word': 'krish so', 'start': 1, 'end': 9}
{'entity_group': 'Email Address', 'score': np.float32(0.7869267), 'word': '##dhi + 91 74991 97208 | sodhi. krish05 @ gmail. com | github / k - r - i - s', 'start': 9, 'end': 71}
{'entity_group': 'Email Address', 'score': np.float32(0.6301086), 'word': 'h -', 'start': 72, 'end': 74}
{'entity_group': 'Degree', 'score': np.float32(0.21623977), 'word': 'b', 'start': 86, 'end': 87}
{'entity_group': 'Degree', 'score': np.float32(0.30545193), 'word': 'tech in', 'start': 88, 'end': 95}
{'entity_group': 'Designation', 'score': np.float32(0.14965703), 'word': 'computer', 'start': 96, 'end': 104}
{'entity_group': 'Degree', 'score': np.float32(0.17503034), 'word': 'engineering', 'start': 105, 'end': 116}
{'entity_group': 'Email Address', 'score': np.float32(0.43126693), 'word': 'aug 2023', 'start': 117, 'end': 125}
{'entity_group': 'Email Address', 'score': np.float32(0.3625806), 'word': 'june', 'st

In [27]:
for r in results:
    print(f"{r['entity_group']:25} → {r['word']} ({r['score']:.2f})")

Name                      → krish so (0.37)
Email Address             → ##dhi + 91 74991 97208 | sodhi. krish05 @ gmail. com | github / k - r - i - s (0.79)
Email Address             → h - (0.63)
Degree                    → b (0.22)
Degree                    → tech in (0.31)
Designation               → computer (0.15)
Degree                    → engineering (0.18)
Email Address             → aug 2023 (0.43)
Email Address             → june (0.36)
Name                      → ##dar (0.31)
College Name              → patel institute of (0.24)
